In [1]:
"""
PIPELINE STAGE: Capacity Dataset Cleaning & Structural Refinement

INPUT: processed_data/steps/01_Capacity.xlsx
OUTPUT: processed_data/steps/02_Capacity_Cleaned.xlsx
LIBRARIES: os, pandas

1. OBJECTIVE:
    Clean the raw installed-capacity master dataset extracted from monthly energy reports.
    The workflow removes unnecessary energy-source columns, eliminates summary records,
    standardizes province names, and prepares the dataset for subsequent harmonization,
    integration, and analytical processing.

2. COLUMN FILTERING & DATASET REFINEMENT:
    A. Column Name Normalization:
       - Standardize column names before comparison.
       - Remove leading/trailing spaces.
       - Convert to lowercase.
       - Normalize Turkish character artifacts:
           replace("i̇", "i")

    B. Unwanted Capacity Sources:
       Remove the following columns if present:

           * doğal gaz
           * linyit
           * toplam

       These fields are excluded because the study focuses on renewable
       installed-capacity indicators rather than fossil-fuel or aggregate totals.

    C. Dynamic Column Detection:
       - Search column names programmatically.
       - Apply deletion regardless of capitalization or spacing differences.

3. PROVINCE COLUMN IDENTIFICATION:
    A. Flexible Detection:
       - Locate the province column using keyword matching.
       - Identify any column containing:

           "iller"

       after normalization.

    B. Validation:
       - Continue province-level cleaning only if a valid province column exists.
       - Log a warning when the province column cannot be found.

4. SPATIAL STANDARDIZATION:
    A. Province Name Cleaning:
       - Remove leading/trailing spaces.
       - Convert all province names to uppercase.

    B. Turkish Character Preservation:
       Apply Turkish-aware uppercase conversion to preserve
       national character consistency.

       Examples:

           izmir      → İZMİR
           istanbul   → İSTANBUL
           kırıkkale  → KIRIKKALE

    C. Consistent Geographic Representation:
       Ensure all provincial records follow a uniform naming convention
       suitable for merging and relational operations.

5. SUMMARY RECORD REMOVAL:
    A. Aggregate Record Detection:
       Identify records where the province field equals:

           GENEL TOPLAM

    B. Dataset Filtering:
       Remove all aggregate rows from the dataset.

    C. Provincial Integrity:
       Retain only province-level observations for statistical analysis.

6. DATA QUALITY CONTROL:
    A. Missing Province Handling:
       - Verify the existence of the province column.
       - Log potential structural issues if absent.

    B. Row Count Validation:
       - Compare dataset size before and after filtering.
       - Report the number of removed summary rows.

7. OUTPUT GENERATION:
    A. Export Clean Dataset:
       Save the refined dataset as:

           processed_data/steps/02_Capacity_Cleaned.xlsx

    B. Export Settings:
       - Excel format (.xlsx)
       - No index column

8. PROCESS LOGGING & MONITORING:
    A. Runtime Messages:
       Display:
           - Source file name
           - Removed columns
           - Province standardization status
           - Number of deleted summary rows

    B. Warning Messages:
       Report missing province columns or structural inconsistencies.

    C. Completion Messages:
       Confirm successful generation of the cleaned capacity dataset.

9. EXPECTED OUTPUT DATASET:
    The resulting dataset contains:

        - Province-level renewable installed-capacity values
        - Standardized uppercase province names
        - Year
        - Month
        - Renewable energy source capacities only

    Fossil-fuel capacity fields and aggregate summary records are excluded,
    producing a cleaner dataset suitable for integration with generation,
    demographic, meteorological, and machine-learning pipelines.
"""

import os
import pandas as pd

# 1. Dosya Yolları
BASE_DIR = r"C:\Users\w11\dergi2"
input_file = os.path.join(BASE_DIR, "processed_data", "steps", "01_capacity.xlsx")

# Temizlenmiş veriyi aynı isimle üzerine yazabilir veya yeni isimle kaydedebiliriz. 
# Güvenlik için "_cleaned" ekiyle kaydedelim, istersen değiştirebilirsin.
output_file = os.path.join(BASE_DIR, "processed_data", "steps", "02_capacity_cleaned.xlsx")

def clean_capacity_data(input_path, output_path):
    print(f">>> Veri okunuyor: {os.path.basename(input_path)}")
    
    # Excel dosyasını oku
    df = pd.read_excel(input_path)
    
    # --- 1, 2, 3: SÜTUN SİLME İŞLEMİ ---
    # Sütun isimlerinde Word'den kalan gizli boşluklar olabilir diye isimleri temizleyip kontrol ediyoruz.
    cols_to_remove = ["doğal gaz", "linyit", "toplam"]
    
    for col in list(df.columns):
        clean_col_name = str(col).strip().lower().replace("i̇", "i")
        if clean_col_name in cols_to_remove:
            df.drop(columns=[col], inplace=True)
            print(f" [✓] Sütun silindi: {col}")

    # --- 4 & 5: İLLER SÜTUNU İŞLEMLERİ (Satır Silme ve Büyük Harf) ---
    # Öncelikle sütun isminin gerçekten "İLLER" veya benzeri olduğundan emin olalım
    iller_col = None
    for col in df.columns:
        if "iller" in str(col).strip().lower().replace("i̇", "i"):
            iller_col = col
            break
            
    if iller_col:
        # Türkçe karakterleri koruyarak büyük harfe çeviren özel fonksiyon
        def turkce_buyuk_harf(metin):
            if pd.isna(metin):
                return metin
            metin = str(metin).strip()
            return metin.replace("i", "İ").replace("ı", "I").upper()
            
        # 5. Adım: Tüm illeri (ve satırları) Türkçe kurallarına göre büyük harf yap
        df[iller_col] = df[iller_col].apply(turkce_buyuk_harf)
        print(" [✓] Bütün iller büyük harfe çevrildi.")

        # 4. Adım: "GENEL TOPLAM" satırını sil
        # Artık hepsi büyük harf olduğu için bulması çok kolay
        before_count = len(df)
        df = df[df[iller_col] != "GENEL TOPLAM"]
        after_count = len(df)
        
        if before_count != after_count:
            print(f" [✓] 'Genel Toplam' satırı silindi (Silinen satır sayısı: {before_count - after_count})")
    else:
        print(" [!] Hata: 'İLLER' sütunu bulunamadı!")

    # Temizlenmiş veriyi kaydet
    df.to_excel(output_path, index=False)
    print(f"\n✅ TEMİZLİK TAMAMLANDI! Dosya kaydedildi: {output_path}")


# --- KODU ÇALIŞTIR ---
if __name__ == "__main__":
    # Eğer dosya adın xlsx değil de csv ise yukarıdaki yolları .csv olarak değiştirmeyi unutma.
    if os.path.exists(input_file):
        clean_capacity_data(input_file, output_file)
    else:
        print(f"Hata: İşlenecek dosya bulunamadı -> {input_file}")

>>> Veri okunuyor: 01_capacity.xlsx
 [✓] Sütun silindi: Doğal Gaz
 [✓] Sütun silindi: Toplam
 [✓] Sütun silindi: Linyit
 [✓] Bütün iller büyük harfe çevrildi.
 [✓] 'Genel Toplam' satırı silindi (Silinen satır sayısı: 60)

✅ TEMİZLİK TAMAMLANDI! Dosya kaydedildi: C:\Users\w11\dergi2\processed_data\steps\02_capacity_cleaned.xlsx
